In [2]:
from typing import TypedDict,Annotated, Optional
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, ToolMessage,AIMessage, BaseMessage
from IPython.display import Image, display
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from ddgs import DDGS
from langgraph.graph.message import add_messages
# LangGraph Advance
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field

load_dotenv()

groq_api = os.getenv("GROQ_API")

In [3]:
llm = ChatGroq(model='llama-3.1-8b-instant',api_key = groq_api)

### LangGraph Basics

#### Define State, Tools and LLM

In [ ]:
class State(TypedDict):
    messages: list

#### Using @tool and ToolNode (LLM-Driven LangGraph System)

In [ ]:
@tool
def get_wiki(query: str,verbose:bool =False):
    """Search Wikipedia for factual information about people, companies, organizations, or topics."""
    if verbose:
        print(f"Query passed to wikipedia:{query}")
    wiki_api = WikipediaAPIWrapper()
    wiki = WikipediaQueryRun(api_wrapper=wiki_api)
    return wiki.run(query)


##### Learning
<br>-> If you notice in the tool defination above you would see 
we didn't pass state instead we passed the query or input the 
tool should be expecting from the LLM as parameter. The Reason 
for this is because there are two approaches in LangGraph for 
binding a tool with an LLM either you can bind it with @tool method (agent Styling) and LangGraph Node Tool (Build for complex Systems). 
Local Documentation Link: Look for Readings/LangGraph Basics journey Document for reading notes
</br>

In [ ]:
tools = [get_wiki]

In [ ]:
llm_wth_tools = llm.bind_tools(tools)

In [ ]:
def llm_call(state:State):
    print()
    messages = state['messages']
    response = llm_wth_tools.invoke(messages)
    return {'messages':messages+[response]}

#### Define Nodes and edges

In [ ]:
tools_node = ToolNode(tools)

In [ ]:
workflow = StateGraph(State)
workflow.add_node('chatbot',llm_call)
workflow.add_node('tool_node',tools_node)
workflow.add_edge(START,'chatbot')
workflow.add_edge("chatbot", "tool_node") # workflow.add_edge("tool_node", "chatbot"): Try running this instead and understand why its important to declare tool node after llm node
# Hint: Edges must form a forward path from START
# Why Does Tool node has an Automatic connection with END Node in chatbot --> toolnode edge and left tool node completely hanging in second case?
# Ans---> LangGraph only “fixes” nodes that are reachable from START and in case two tool node is not reachable from start. so langGraph doesn't know what to do.
workflow.add_edge('chatbot',END)
app = workflow.compile()

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
response = app.invoke(
    {
        "messages": [HumanMessage(content="Tell me about NASA")]
    }
)

print(response["messages"][-1].content)

#### LangGraph Node Tool

In [ ]:
def wiki_node(state: State):
    print('============Wikipedia Tool Call=================')

    query = state["messages"][-1].content

    try:
        result = wiki.run(query)
    except Exception as e:
        result = f"Wiki search failed: {str(e)}"

    return {
        "messages": state["messages"] + [
            AIMessage(content=f"Wiki Result:\n{result}")  # Don't USe  ToolMessage its internal to tool node.
        ]
    }

In [ ]:
def web_search_node(state:State):
    print('============Duck Duck Go Tool Call=================')
    query = state["messages"][-1].content

    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=3)

    formatted = "\n\n".join(
        [f"{r['title']}\n{r['body']}" for r in results]
    )

    return {
        "messages":  state["messages"] + [
            AIMessage(content=f"Web Search Results:\n\n{formatted}") # Don't USe  ToolMessage its internal to tool node.
        ]
    }

In [ ]:
def router(state):
    last_message = state["messages"][-1].content.lower()

    if "who is" in last_message or "what is" in last_message:
        return "wiki" # Should be same as the name of key declared in conditional edge while creating Graph

    if "latest" in last_message or "news" in last_message or "Tell me" in last_message:
        return "web_search" # Should be same as the name of key declared in conditional edge while creating Graph

    return "end"

In [ ]:
def llm_call_new(state:State):
    messages = state['messages']
    print(f"LLM Called with {messages}[-1]")
    response = llm.invoke(messages)
    return {'messages':messages+[response]}

In [ ]:
from langgraph.graph import StateGraph, START, END

multitool_workflow = StateGraph(State)

multitool_workflow.add_node("chatbot", llm_call_new)
multitool_workflow.add_node("wiki", wiki_node)
multitool_workflow.add_node("web_search", web_search_node)

multitool_workflow.add_edge(START, "chatbot")

multitool_workflow.add_conditional_edges(
    "chatbot",
    router,
    {
        "wiki": "wiki",
        "web_search": "web_search",
        "end": END
    }
)
# Conditonal Edge Syntax:
# workflow.add_conditional_edges(
#     from_node,
#     condition_function,
#     mapping_dict
# )
# If you dont declare all the return output of router in mapping_dict then your llm is going to through a runtime error as (say you didn't may end output value of router to any node)
# you will get a ValueError: No edge found for condition: 'end'
# and if you don't have a end as retun output of your router then your graph is going to go in an infinite loop stuck between tool calls and nowhere to got to for ending the iteration.
multitool_app = multitool_workflow.compile()

In [ ]:


display(Image(multitool_app.get_graph().draw_mermaid_png()))

In [ ]:
response = multitool_app.invoke(
    {
        "messages": [HumanMessage(content="Latest in Artifical Inteligence")]
    }
)

print(response["messages"][-1].content)

### LangGraph Advance

#### 1.Adding Memory

In [6]:
memory = MemorySaver()

In [7]:
class State(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]

In [8]:
def llm_call_memory(state: State):
    messages = state["messages"]
    
    response = llm.invoke(messages)
    
    return {
        "messages": [response] 
    }

In [ ]:
workflow_wth_memory = StateGraph(State)

workflow_wth_memory.add_node('llm_call',llm_call_memory)

workflow_wth_memory.add_edge(START, 'llm_call')
workflow_wth_memory.add_edge('llm_call',END)

app_wth_memory = workflow_wth_memory.compile(checkpointer=memory)

display(Image(app_wth_memory.get_graph().draw_mermaid_png()))

In [ ]:
thread_config = {"configurable":
                 {'thread_id':'user_1'}
                 }

In [ ]:
response = app_wth_memory.invoke({
    'messages':[HumanMessage(content="Hi My Name is Abhishek")]
},config = thread_config)


In [ ]:
for messages in response['messages']:
    messages.pretty_print()

In [ ]:
response = app_wth_memory.invoke({
    'messages':[HumanMessage(content="Hi, What is my name?")]
},config = thread_config)

In [ ]:
for messages in response['messages']:
    messages.pretty_print()

##### Breaking Memory

In [ ]:
thread_config2 = {"configurable":
                 {'thread_id':'user_2'}
                 }

In [ ]:
response = app_wth_memory.invoke({
    'messages':[HumanMessage(content="Hi, What is my name?")]
},config = thread_config2)


In [ ]:
for messages in response['messages']:
    messages.pretty_print()

In [ ]:
response = app_wth_memory.invoke({
    'messages':[HumanMessage(content="Hey, What is my name and what do i do?")]
},config = thread_config)

for messages in response['messages']:
    messages.pretty_print()

##### QA:
<br>
1. Understand What distribute attention across tokens mean?

Ans. “Distribute attention across tokens” means the model spreads its focus across all input, so more tokens = weaker focus on each → important info gets diluted. For a deep dive understand Transformenrs Architecture.
</br>
<br>
2. Understand how real systems solve memory problems (summary memory + retrieval memory).
</br>

##### LLM With Windowed memory

In [12]:
def llm_wth_window_memory(state: State):
    messages = state['messages'][-2:] # Last 3 Messages
    response = llm.invoke(messages)
    return {
        'messages': [response]
    }

In [ ]:
memory = MemorySaver()

session = {
    "configurable":{
        'thread_id':'user001'
    }
}

In [ ]:
workflow_wth_window = StateGraph(State)
workflow_wth_window.add_node('llm_call',llm_wth_window_memory)

workflow_wth_window.add_edge(START, 'llm_call')
workflow_wth_window.add_edge('llm_call',END)
app_wth_window = workflow_wth_memory.compile(checkpointer = memory)

In [ ]:
display(Image(app_wth_window.get_graph().draw_mermaid_png()))

In [ ]:
while True:
    query = str(input("Type your message"))
    if query in ['bye','quit']:
        break
    else:
        response = app_wth_window.invoke({
            'messages': HumanMessage(content=query)
        },config = session)
        for msg in response['messages']:
            msg.pretty_print()


In [ ]:
print(len(response["messages"]))

In [ ]:
response = app_wth_window.invoke({
    'messages': HumanMessage(content='Okay but do you know where i live?')
},config = session)

In [ ]:
for msg in response['messages'][-3:]:
    msg.pretty_print()

##### Learning:
<br> 
There are 3 Types of Memory for LLM:
    
    1. Short Term Memory (we just used): Providing only a few messages for follow up questions. for example [-5:],[-10] (providing last 5 or 10 messages from the conversation history).
    
    2. Long Term Memory: Porviding a summary of the current session for better context but doent include the actual facts.
    
    3. Structured Memory: Keeping facts such as user profiles, Likes dislikes, Job preferences, Interests, Personality traits except
</br>

##### 1.2. Structured memory

In [27]:
class User(BaseModel):
    name: Optional[str] = Field(None, description="Name of the person")
    city: Optional[str] = Field(None, description="City the person lives in")
    profession: Optional[str] = Field(None, description="Profession of the person")

In [28]:
structured_llm = llm.with_structured_output(User)

In [40]:
user_profile = {}
def struct_memory(state:State):
    """Useful for storing and creating User profiles base on Conversation chat LLM is having"""
    print("=========Struct Memory Called========")
    message = state['messages'][-1].content
    response = structured_llm.invoke(f"""
        Extract ONLY information about the USER (speaker).
        Ignore references to other people like 'you', 'he', 'she', 'friend'.
        Text: {message}
        """)
    print(f"Response from Struct LLM: {response}") 
    if response.name and response.name.lower() not in ["you", "he", "she", "they"]:
        user_profile['name'] = response.name
    if response.city:
        user_profile['city'] = response.city
    if response.profession:
        user_profile['profession'] = response.profession
    
    return state

In [41]:

session_2 = {"configurable":
                {'thread_id':'user002'}}

In [42]:
"""Adding facts to LLM memory for better interaction experience"""
def struct_memory_keyword(state:State):
    global user_profile

    print("struct_memory Called")
    last_message = state['messages'][-1].content.lower()

    if "my name is" in last_message:
        user_profile['name']= last_message.split()[-1].strip().title()
    elif "I am a" in last_message:
        user_profile['profession'] = last_message.split()[-1].strip().title()
        return state

In [43]:
workflow_struct = StateGraph(State)
workflow_struct.add_node('extract_structured_memory',struct_memory)
workflow_struct.add_node('llm_call_struct',llm_wth_window_memory)

workflow_struct.add_edge(START,'extract_structured_memory')
workflow_struct.add_edge('extract_structured_memory','llm_call_struct')
workflow_struct.add_edge('llm_call_struct',END)

app_wth_stct = workflow_struct.compile(checkpointer = memory)

In [44]:
while True:
    query = str(input("Type your message"))
    if query in ['bye','quit']:
        break
    else:
        response = app_wth_stct.invoke({
            'messages': HumanMessage(content=query)
        },config = session_2)
        for msg in response['messages']:
            msg.pretty_print()


=========Struct Memory Called========
Response from Struct LLM: name='abhishek' city=None profession=None
================================ Human Message =================================

Hi My Name is Abhishek
================================== Ai Message ==================================

Nice to meet you Abhishek. Is there something I can help you with or would you like to have a conversation?
================================ Human Message =================================

Sure, as i am a data scientist can you summaries the future of data science keeping AI in equation?
================================== Ai Message ==================================

As a data scientist, you're at the forefront of the exciting field that's rapidly evolving with the integration of AI. Here's a summary of the future of data science, incorporating AI:

**Near-term trends (2023-2025)**

1. **Increased adoption of Explainable AI (XAI)**: As AI becomes more pervasive, there's a growing need to understa

In [45]:
print(user_profile)

{'name': 'abhishek'}
